In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os

2025-03-17 14:48:07.803907: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742222887.814932  133035 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742222887.818406  133035 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1742222887.827839  133035 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742222887.827848  133035 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742222887.827850  133035 computation_placer.cc:177] computation placer alr

In [3]:
os.getcwd()

'/home/selc-a4-sr2/Solar_Rooftop_Detection/BaseLineModels/Unet_model'

In [4]:
os.chdir("../")
os.getcwd()

'/home/selc-a4-sr2/Solar_Rooftop_Detection/BaseLineModels'

In [5]:

root_dir = '/home/selc-a4-sr2/Solar_Rooftop_Detection'

In [6]:
# Load preprocessed training images and labels
train_input_dir_images = root_dir + '/Arial_images_1024_1024/images'
train_input_dir_masks = root_dir + '/Arial_images_1024_1024/masks'

X_train = []
y_train = []

for i in range(int(len(os.listdir(train_input_dir_images)))):
    image_filename = f"{os.listdir(train_input_dir_images)[i]}"
    image_path = os.path.join(train_input_dir_images, image_filename)
    image = cv2.imread(image_path)
    X_train.append(image)

    label_filename = f"{os.listdir(train_input_dir_images)[i]}"
    label_path = os.path.join(train_input_dir_masks, label_filename)
    label = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)
    y_train.append(label)

# Load preprocessed validation images and labels
val_input_dir_images = root_dir + '/Arial_validation_images/images'
val_input_dir_masks = root_dir + '/Arial_validation_images/masks'


X_test = []
y_test = []

for i in range(int(len(os.listdir(val_input_dir_images)))):
    image_filename = f"{os.listdir(val_input_dir_images)[i]}"
    image_path = os.path.join(val_input_dir_images, image_filename)
    image = cv2.imread(image_path)
    X_test.append(image)

    label_filename = f"{os.listdir(val_input_dir_images)[i]}"
    label_path = os.path.join(val_input_dir_masks, label_filename)
    label = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)
    y_test.append(label)

# Load preprocessed test images
test_input_dir = root_dir + "/Arial_test_images/images"

test_images = []

for i in range(int(len(os.listdir(test_input_dir)))):
    image_filename = f"{i}.jpg"
    image_path = os.path.join(test_input_dir, image_filename)
    image = cv2.imread(image_path)
    test_images.append(image)

In [12]:
# Normalize pixel values
train_images = np.array(train_images) / 255.0
train_labels = np.array(train_labels) / 255.0

val_images = np.array(val_images) / 255.0
val_labels = np.array(val_labels) / 255.0

test_images = np.array(test_images) / 255.0

In [13]:
# train_images = tf.convert_to_tensor(train_images)
# train_labels = tf.convert_to_tensor(train_labels)

# val_images = tf.convert_to_tensor(val_images)
# val_labels = tf.convert_to_tensor(val_labels)

# test_images = tf.convert_to_tensor(test_images)

In [14]:
train_images.shape

(2304, 1024, 1024, 3)

In [15]:
from tensorflow.keras import layers

In [16]:
def unet():
    inputs = keras.Input(shape=(None, None, 3))

    # Contracting Path
    conv1 = layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
    conv1 = layers.Conv2D(64, 3, activation='relu', padding='same')(conv1)
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = layers.Conv2D(128, 3, activation='relu', padding='same')(pool1)
    conv2 = layers.Conv2D(128, 3, activation='relu', padding='same')(conv2)
    pool2 = layers.MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = layers.Conv2D(256, 3, activation='relu', padding='same')(pool2)
    conv3 = layers.Conv2D(256, 3, activation='relu', padding='same')(conv3)
    pool3 = layers.MaxPooling2D(pool_size=(2, 2))(conv3)

    # Bottleneck
    conv4 = layers.Conv2D(512, 3, activation='relu', padding='same')(pool3)
    conv4 = layers.Conv2D(512, 3, activation='relu', padding='same')(conv4)

    # Expansive Path
    up1 = layers.Conv2DTranspose(256, 2, strides=(2, 2), padding='same')(conv4)
    up1 = layers.concatenate([up1, conv3])
    conv5 = layers.Conv2D(256, 3, activation='relu', padding='same')(up1)
    conv5 = layers.Conv2D(256, 3, activation='relu', padding='same')(conv5)

    up2 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(conv5)
    up2 = layers.concatenate([up2, conv2])
    conv6 = layers.Conv2D(128, 3, activation='relu', padding='same')(up2)
    conv6 = layers.Conv2D(128, 3, activation='relu', padding='same')(conv6)

    up3 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(conv6)
    up3 = layers.concatenate([up3, conv1])
    conv7 = layers.Conv2D(64, 3, activation='relu', padding='same')(up3)
    conv7 = layers.Conv2D(64, 3, activation='relu', padding='same')(conv7)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(conv7)

    return keras.Model(inputs=inputs, outputs=outputs)

In [17]:
optimizer = 'adam' # good for boundary detection; other options: RMSprop, SGD
loss = 'binary_crossentropy'
metrics = ['binary_accuracy'] # other options: binary_crossentropy

In [18]:
# with tf.device('/GPU:0'):
model = unet()
model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
model.summary()

2025-03-17 09:55:11.685523: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-03-17 09:55:11.685547: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:167] env: CUDA_VISIBLE_DEVICES="-1"
2025-03-17 09:55:11.685551: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2025-03-17 09:55:11.685554: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-03-17 09:55:11.685557: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: SELC-A4-SRV01
2025-03-17 09:55:11.685559: I external/local_xla/xla/stream_executor/cuda

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, None,      │      1,792 │ input_layer[0][0] │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, None,      │     36,928 │ conv2d[0][0]      │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, None,      │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, None,      │     73,856 │ max_pooling2d[0]… │
│                     │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, None,      │    147,584 │ conv2d_2[0][0]    │
│                     │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, None,      │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, None,      │    295,168 │ max_pooling2d_1[… │
│                     │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, None,      │    590,080 │ conv2d_4[0][0]    │
│                     │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, None,      │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, None,      │  1,180,160 │ max_pooling2d_2[… │
│                     │ None, 512)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, None,      │  2,359,808 │ conv2d_6[0][0]    │
│                     │ None, 512)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, None,      │    524,544 │ conv2d_7[0][0]    │
│ (Conv2DTranspose)   │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, None,      │          0 │ conv2d_transpose… │
│ (Concatenate)       │ None, 512)        │            │ conv2d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, None,      │  1,179,904 │ concatenate[0][0] │
│                     │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, None,      │    590,080 │ conv2d_8[0][0]    │
│                     │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_1  │ (None, None,      │    131,200 │ conv2d_9[0][0]  

 Total params: 7,697,345 (29.36 MB)

 Trainable params: 7,697,345 (29.36 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
batch_size = 8
epochs = 10

In [24]:
# Assuming you have loaded your training data into `train_images` and `train_masks` arrays
history = model.fit(train_images, train_labels, batch_size=batch_size, epochs=epochs)

: 

In [ ]:
train_loss = history.history['loss']
val_loss = history.history['val_loss']

c = ['r', 'k']
lw, ms, marker = 2, 3, ''

# Plotting the loss curve
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(train_loss, label='Training Loss', color=c[0], marker=marker, ms=ms, lw=lw)
ax.plot(val_loss, label='Validation Loss', color=c[1], marker=marker, ms=ms, lw=lw)
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.spines[['right', 'top']].set_visible(False)
ax.legend(frameon=False)
ax.grid()
plt.tight_layout()
plt.show()

# Plotting the Metrics curve
fig, ax = plt.subplots(figsize=(10,6))
for metric in metrics:
  train_metrics = history.history[f'{metric}']
  val_metrics = history.history[f'val_{metric}']

  ax.plot(train_metrics, label=f'Training {metrics[0]}', color=c[0], marker=marker, ms=ms, lw=lw)
  ax.plot(val_metrics, label=f'Validation {metrics[0]}', color=c[1], marker=marker, ms=ms, lw=lw)
  ax.set_xlabel('Epochs')
  ax.set_ylabel(f'{metrics}')
  ax.spines[['right', 'top']].set_visible(False)
  ax.legend(frameon=False)

ax.grid()
ax.set_ylim(bottom=0.85)
plt.tight_layout()
plt.show()

In [ ]:
val_loss, val_accuracy = model.evaluate(val_images, val_labels)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
val_predictions = model.predict(val_images)

In [ ]:
# Apply thresholding to convert predicted masks to binary images
def post_process(predictions, threshold=0.5):
    binary_images = (predictions > threshold).astype(np.uint8)
    return binary_images

val_processed_predictions = post_process(val_predictions, threshold=0.5)

In [ ]:
fig, axes = plt.subplots(10, 3, figsize=(20, 50))

for i, idx in enumerate(np.argsort(accuracies)[::-1][:10]):

    # Display test image
    axes[i, 0].imshow(val_images[idx])
    axes[i, 0].set_title('Validation Image')
    axes[i, 0].axis('off')

    # Display prediction
    axes[i, 1].imshow(val_processed_predictions[idx], cmap='gray')
    axes[i, 1].set_title('Binarised Validation Prediction')
    axes[i, 1].axis('off')

    # Display prediction with post-processed mask
    axes[i, 2].imshow(val_labels[idx], cmap='gray')
    axes[i, 2].set_title('Validation Label')
    axes[i, 2].axis('off')


plt.tight_layout()
plt.show()

In [ ]:
# Save the entire model
model.save("/home/selc-a4-sr2/Solar_Rooftop_Detection/BaseLineModels/Unet_model/UNET_MODEL.h5")

# Load the saved model
#from tensorflow.keras.models import load_model
#model = load_model("../models/baseline.h5")

In [20]:
X_train = np.array(X_train) / 255.0
X_test = np.array(X_test) / 255.0
y_train = np.array(y_train) / 255.0
y_test = np.array(y_test) / 255.0 

In [22]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((2304, 1024, 1024, 3),
 (576, 1024, 1024, 3),
 (2304, 1024, 1024),
 (576, 1024, 1024))

In [8]:
os.getcwd()

'/home/selc-a4-sr2/Solar_Rooftop_Detection/BaseLineModels'

In [23]:
from Unet_model.simple_unet_model import simple_unet_model   #Use normal unet model
from keras.utils import normalize
import os
import cv2
from PIL import Image
import numpy as np
from matplotlib import pyplot as plt

In [24]:



# image_directory = 'data/generated_patches/images/'
# mask_directory = 'data/generated_patches/masks/'


# SIZE = 256
# image_dataset = []  #Many ways to handle data, you can use pandas. Here, we are using a list format.  
# mask_dataset = []  #Place holders to define add labels. We will add 0 to all parasitized images and 1 to uninfected.

# images = os.listdir(image_directory)
# for i, image_name in enumerate(images):    #Remember enumerate method adds a counter and returns the enumerate object
#     if (image_name.split('.')[1] == 'tif'):
#         #print(image_directory+image_name)
#         image = cv2.imread(image_directory+image_name, 0)
#         image = Image.fromarray(image)
#         image = image.resize((SIZE, SIZE))
#         image_dataset.append(np.array(image))

# #Iterate through all images in Uninfected folder, resize to 64 x 64
# #Then save into the same numpy array 'dataset' but with label 1

# masks = os.listdir(mask_directory)
# for i, image_name in enumerate(masks):
#     if (image_name.split('.')[1] == 'tif'):
#         image = cv2.imread(mask_directory+image_name, 0)
#         image = Image.fromarray(image)
#         image = image.resize((SIZE, SIZE))
#         mask_dataset.append(np.array(image))


#Normalize images
# image_dataset = np.expand_dims(normalize(np.array(image_dataset), axis=1),3)
# #D not normalize masks, just rescale to 0 to 1.
# mask_dataset = np.expand_dims((np.array(mask_dataset)),3) /255.


#Sanity check, view few mages
# import random
# import numpy as np
# image_number = random.randint(0, len(X_train))
# plt.figure(figsize=(12, 6))
# plt.subplot(121)
# plt.imshow(np.reshape(X_train[image_number], (256, 256)), cmap='gray')
# plt.subplot(122)
# plt.imshow(np.reshape(y_train[image_number], (256, 256)), cmap='gray')
# plt.show()

###############################################################
IMG_HEIGHT = 1024
IMG_WIDTH  = 1024
IMG_CHANNELS = 3

def get_model():
    return simple_unet_model(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

model = get_model()


#If starting with pre-trained weights. 
#model.load_weights('mitochondria_gpu_tf1.4.hdf5')

history = model.fit(X_train, y_train, 
                    batch_size = 16, 
                    verbose=1, 
                    epochs=5, 
                    validation_data=(X_test, y_test), 
                    shuffle=False)

model.save('Unet_model_1_epoch_5.hdf5')

############################################################
#Evaluate the model


	# evaluate model
_, acc = model.evaluate(X_test, y_test)
print("Accuracy = ", (acc * 100.0), "%")


#plot the training and validation accuracy and loss at each epoch
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'y', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

acc = history.history['acc']
#acc = history.history['accuracy']
val_acc = history.history['val_acc']
#val_acc = history.history['val_accuracy']

plt.plot(epochs, acc, 'y', label='Training acc')
plt.plot(epochs, val_acc, 'r', label='Validation acc')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

##################################
#IOU
y_pred=model.predict(X_test)
y_pred_thresholded = y_pred > 0.5

intersection = np.logical_and(y_test, y_pred_thresholded)
union = np.logical_or(y_test, y_pred_thresholded)
iou_score = np.sum(intersection) / np.sum(union)
print("IoU socre is: ", iou_score)



I0000 00:00:1742223398.807267  133035 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 44829 MB memory:  -> device: 0, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1024,      │          0 │ -                 │
│ (InputLayer)        │ 1024, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 1024,      │        448 │ input_layer[0][0] │
│                     │ 1024, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1024,      │          0 │ conv2d[0][0]      │
│                     │ 1024, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 1024,      │      2,320 │ dropout[0][0]     │
│                     │ 1024, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 512, 512,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 512, 512,  │      4,640 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512, 512,  │          0 │ conv2d_2[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 512, 512,  │      9,248 │ dropout_1[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 256, 256,  │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 256, 256,  │     18,496 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256, 256,  │          0 │ conv2d_4[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 256, 256,  │     36,928 │ dropout_2[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 128, 128,  │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 128, 128,  │     73,856 │ max_pooling2d_2[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128, 128,  │          0 │ conv2d_6[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 128, 128,  │    147,584 │ dropout_3[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 64, 64,    │          0 │ conv2d_7[0][0]  

 Total params: 1,941,105 (7.40 MB)

 Trainable params: 1,941,105 (7.40 MB)

 Non-trainable params: 0 (0.00 B)

2025-03-17 14:57:41.796591: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 28991029248 exceeds 10% of free system memory.
2025-03-17 14:58:13.913351: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 9663676416 exceeds 10% of free system memory.
2025-03-17 14:58:22.586143: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 28991029248 exceeds 10% of free system memory.
2025-03-17 14:58:34.794692: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 9663676416 exceeds 10% of free system memory.


Epoch 1/5


/home/selc-a4-sr2/myenv/lib/python3.12/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(16, 1024, 1024, 3))
  warnings.warn(msg)
I0000 00:00:1742223525.342968  152291 service.cc:152] XLA service 0x759734009a60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1742223525.344180  152291 service.cc:160]   StreamExecutor device (0): NVIDIA RTX 6000 Ada Generation, Compute Capability 8.9
2025-03-17 14:58:45.505110: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
E0000 00:00:1742223526.100295  152291 cuda_dnn.cc:522] Loaded runtime CuDNN library: 9.1.0 but source was compiled with: 9.3.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, u

FailedPreconditionError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/selc-a4-sr2/myenv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/selc-a4-sr2/myenv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/selc-a4-sr2/myenv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/selc-a4-sr2/myenv/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 641, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1986, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/home/selc-a4-sr2/myenv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 54

In [ ]:
#######################################################################
#Predict on a few images
model = get_model()
model.load_weights('mitochondria_50_plus_100_epochs.hdf5') #Trained for 50 epochs and then additional 100
#model.load_weights('mitochondria_gpu_tf1.4.hdf5')  #Trained for 50 epochs

test_img_number = random.randint(0, len(X_test))
test_img = X_test[test_img_number]
ground_truth=y_test[test_img_number]
test_img_norm=test_img[:,:,0][:,:,None]
test_img_input=np.expand_dims(test_img_norm, 0)
prediction = (model.predict(test_img_input)[0,:,:,0] > 0.2).astype(np.uint8)

test_img_other = cv2.imread('data/test_images/02-1_256.tif', 0)
#test_img_other = cv2.imread('data/test_images/img8.tif', 0)
test_img_other_norm = np.expand_dims(normalize(np.array(test_img_other), axis=1),2)
test_img_other_norm=test_img_other_norm[:,:,0][:,:,None]
test_img_other_input=np.expand_dims(test_img_other_norm, 0)

#Predict and threshold for values above 0.5 probability
#Change the probability threshold to low value (e.g. 0.05) for watershed demo.
prediction_other = (model.predict(test_img_other_input)[0,:,:,0] > 0.2).astype(np.uint8)

plt.figure(figsize=(16, 8))
plt.subplot(231)
plt.title('Testing Image')
plt.imshow(test_img[:,:,0], cmap='gray')
plt.subplot(232)
plt.title('Testing Label')
plt.imshow(ground_truth[:,:,0], cmap='gray')
plt.subplot(233)
plt.title('Prediction on test image')
plt.imshow(prediction, cmap='gray')
plt.subplot(234)
plt.title('External Image')
plt.imshow(test_img_other, cmap='gray')
plt.subplot(235)
plt.title('Prediction of external Image')
plt.imshow(prediction_other, cmap='gray')
plt.show()

#plt.imsave('input.jpg', test_img[:,:,0], cmap='gray')
#plt.imsave('data/results/output2.jpg', prediction_other, cmap='gray')


In [1]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

In [2]:
os.getcwd()

'/home/selc-a4-sr2/Solar_Rooftop_Detection/BaseLineModels/Unet_model'

In [3]:
torch.cuda.empty_cache()
torch.backends.cudnn.benchmark = True  # Optimize backend performance
torch.cuda.reset_peak_memory_stats()

In [ ]:
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Paths
root_dir = '/home/selc-a4-sr2/Solar_Rooftop_Detection'
train_images_dir = os.path.join(root_dir, 'Arial_images_1024_1024/images')
train_masks_dir = os.path.join(root_dir, 'Arial_images_1024_1024/masks')
val_images_dir = os.path.join(root_dir, 'Arial_validation_images/images')
val_masks_dir = os.path.join(root_dir, 'Arial_validation_images/masks')

# Custom Dataset
class RooftopDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_filenames = sorted(os.listdir(image_dir))
        self.transform = transform
    
    def __len__(self):
        return len(self.image_filenames)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_filenames[idx])
        mask_path = os.path.join(self.mask_dir, self.image_filenames[idx])
        
        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = mask.astype(np.float32) / 255.0
        
        if self.transform:
            image = self.transform(image)
            mask = transforms.ToTensor()(mask).unsqueeze(0)  # Ensure mask shape [1, H, W]
        
        return image, mask.squeeze(0)  # Ensure mask shape [H, W]

# Transformations
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((1024, 1024)),
    transforms.ToTensor()
])

# Load datasets
train_dataset = RooftopDataset(train_images_dir, train_masks_dir, transform)
val_dataset = RooftopDataset(val_images_dir, val_masks_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

# U-Net Model
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()
        
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.ReLU()
            )
        
        self.encoder = nn.ModuleList([
            conv_block(3, 16),
            conv_block(16, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256)
        ])
        
        self.pool = nn.MaxPool2d(2)
        
        self.upconv = nn.ModuleList([
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        ])
        
        self.decoder = nn.ModuleList([
            conv_block(256, 128),
            conv_block(128, 64),
            conv_block(64, 32),
            conv_block(32, 16)
        ])
        
        self.final_conv = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        enc_outs = []
        for enc in self.encoder:
            x = enc(x)
            enc_outs.append(x)
            x = self.pool(x)
        
        for i in range(4):
            x = self.upconv[i](x)
            enc_out = enc_outs[-(i+2)]
            if x.shape[2:] != enc_out.shape[2:]:
                x = nn.functional.interpolate(x, size=enc_out.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, enc_out], dim=1)
            x = self.decoder[i](x)
        
        return torch.sigmoid(self.final_conv(x))

# Training Setup
model = UNet().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training Loop
def train(model, train_loader, val_loader, epochs=50):
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)  # Ensure correct shape [B, 1, H, W]
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(train_loader):.4f}")
    
    torch.save(model.state_dict(), "unet_rooftop.pth")
    print("Model saved!")

train(model, train_loader, val_loader,epochs=250)

# Evaluation
def evaluate(model, val_loader):
    model.eval()
    iou_scores = []
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            preds = (outputs > 0.5).float()
            
            intersection = (preds * masks).sum()
            union = (preds + masks).sum() - intersection
            iou = intersection / (union + 1e-6)
            iou_scores.append(iou.item())
    
    print(f"Mean IoU Score: {np.mean(iou_scores):.4f}")

evaluate(model, val_loader)

Using device: cuda
Epoch 1/50, Loss: 0.5332
Epoch 2/50, Loss: 0.4198
Epoch 3/50, Loss: 0.3664
Epoch 4/50, Loss: 0.3346
Epoch 5/50, Loss: 0.3108
Epoch 6/50, Loss: 0.2978
Epoch 7/50, Loss: 0.2890
Epoch 8/50, Loss: 0.2731
Epoch 9/50, Loss: 0.2646
Epoch 10/50, Loss: 0.2532
Epoch 11/50, Loss: 0.2452
Epoch 12/50, Loss: 0.2408
Epoch 13/50, Loss: 0.2324
Epoch 14/50, Loss: 0.2273
Epoch 15/50, Loss: 0.2210
Epoch 16/50, Loss: 0.2170
Epoch 17/50, Loss: 0.2125
Epoch 18/50, Loss: 0.2101
Epoch 19/50, Loss: 0.2022
Epoch 20/50, Loss: 0.2009
Epoch 21/50, Loss: 0.1984
Epoch 22/50, Loss: 0.1936
Epoch 23/50, Loss: 0.1892
Epoch 24/50, Loss: 0.1859
Epoch 25/50, Loss: 0.1819
Epoch 26/50, Loss: 0.1789
Epoch 27/50, Loss: 0.1762
Epoch 28/50, Loss: 0.1745
Epoch 29/50, Loss: 0.1722
Epoch 30/50, Loss: 0.1715
Epoch 31/50, Loss: 0.1700
Epoch 32/50, Loss: 0.1677
Epoch 33/50, Loss: 0.1638
Epoch 34/50, Loss: 0.1607
Epoch 35/50, Loss: 0.1609
Epoch 36/50, Loss: 0.1626
Epoch 37/50, Loss: 0.1561
Epoch 38/50, Loss: 0.1570
Ep